# Reproduce `chain-9b-temp-lo-seed-random-rep3`

Generated from `<inference page: chain-9b-temp-lo-seed-random-rep3 x 5 instance(s)>`. The notebook replays inference + evaluation against the exact config snapshot that produced the original run, so a re-execution should produce a comparable `model_patch` (LLM determinism caveats notwithstanding).

- **Instances:** 5
- **Config id:** `chain-9b-temp-lo-seed-random-rep3`


## 1. Setup

The notebook's `kernelspec.name = "evomas"` (see metadata at the bottom of the file) tells Jupyter / VSCode to auto-pick the interpreter `setup.ps1` / `setup.sh` registered for `~/.evomas-venv`. As a safety net the first cell also prepends the venv's site-packages to `sys.path` — so even if the kernel falls back to a generic Python 3 (different machine, no `setup.ps1` run), the evomas imports still resolve. Adjust `OLLAMA_BASE_URL` if your Ollama daemon isn't on the default host; `SWEBENCH_API_KEY` is only required by the remote-eval cell at the bottom.

### Picking the kernel in VSCode

If VSCode opens the notebook outside the EvoMas workspace (e.g. straight from `~/Downloads`), it won't auto-resolve the kernelspec and asks you to **Select Kernel**. Two-tier picker:

- **"Python Environments…"** lists raw Python interpreters discovered by the Python extension (system Python, conda envs, `.venv`/`venv` folders inside workspaces). `~/.evomas-venv` is outside the conventional discovery paths, so it does NOT show up here.
- **"Jupyter Kernel…"** lists registered Jupyter kernelspecs (`%APPDATA%\jupyter\kernels\*` on Windows, `~/.local/share/jupyter/kernels/*` on Linux/mac). This is where the EvoMas one lives — pick **"Python 3 (EvoMas)"** here. VSCode remembers the choice per-notebook so you only have to do it once.

If the entry doesn't appear there: `Ctrl+Shift+P` → **"Developer: Reload Window"** so the Jupyter extension re-scans kernelspecs, or run `jupyter kernelspec list` to confirm `evomas` is registered (if not, re-run `setup.ps1` / `setup.sh`).

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path

# Defensive sys.path prepend so `import evomas...` still
# resolves when the kernel isn't the evomas-venv one.
_venv = Path.home() / '.evomas-venv'
if _venv.is_dir() and str(_venv) not in sys.executable:
    for _sp in (_venv / 'Lib' / 'site-packages',
                _venv / 'lib' / 'site-packages'):
        if _sp.is_dir() and str(_sp) not in sys.path:
            sys.path.insert(0, str(_sp))

import evomas.paths  # noqa: F401  # triggers load_dotenv(evomas/.env)

from evomas.core.workflow.runner import run as run_evomas
from evomas.utils.instances import fetch_swebench_instances

# Mirror logging to a per-run text file so generate_report.py
# can mine the same lines the API matrix path writes.
import logging
RUN_OUTPUT_DIR = Path('notebook-chain-9b-temp-lo-seed-random-rep3').resolve()
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_OUTPUT_DIR / 'inference.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    force=True,
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
    ],
)
print(f'Mirroring inference logs to {LOG_FILE}')


Mirroring inference logs to C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3\inference.log


### Environment variables

All run-time configuration the notebook needs lives in this cell — edit values here rather than chasing them through the code. Each assignment **overrides** whatever is in the environment / .env when the cell runs.

- **`OLLAMA_BASE_URL`** — where the Ollama daemon serves.
- **`SWEBENCH_API_KEY`** — required by the remote-eval cell (section 5) when running `--remote` against sb-cli. Local Docker harness runs don't need it.
- **`EVOMAS_INSTANCES`** — override the SWE-bench instance cache location. The next cell already searches sensible defaults; only set this if your cache is somewhere non-standard.
- **`GOOGLE_API_KEY` / `OPENAI_API_KEY`** — only needed if the inlined config picks a Gemini / OpenAI model instead of Ollama.


In [2]:
# Edit values here; each assignment overrides the inherited
# environment / .env. Uncomment the lines you need.
os.environ['OLLAMA_BASE_URL'] = 'http://192.168.1.50:11434'
# os.environ['SWEBENCH_API_KEY'] = 'swb_...'
# os.environ['EVOMAS_INSTANCES'] = '/path/to/swebench_instances.jsonl'
# os.environ['GOOGLE_API_KEY']   = '...'
# os.environ['OPENAI_API_KEY']   = '...'

# Echo the effective values (mask secrets) so you can verify the cell ran.
for _k in ('OLLAMA_BASE_URL', 'SWEBENCH_API_KEY', 'EVOMAS_INSTANCES',
           'GOOGLE_API_KEY', 'OPENAI_API_KEY'):
    _v = os.environ.get(_k, '')
    if not _v:
        print(f'  {_k:<18} <unset>')
    elif _k.endswith('_API_KEY'):
        print(f'  {_k:<18} {_v[:6]}***({len(_v)} chars)')
    else:
        print(f'  {_k:<18} {_v}')


  OLLAMA_BASE_URL    http://192.168.1.50:11434
  SWEBENCH_API_KEY   swb_QM***(56 chars)
  EVOMAS_INSTANCES   <unset>
  GOOGLE_API_KEY     AIzaSy***(39 chars)
  OPENAI_API_KEY     <unset>


## 2. Inlined config

Exact resolved config the original run used. Tweak hyperparameters here if you want to experiment with variations.

The cell below the config dict renders a mermaid diagram of the topology so you can see the agent-graph shape at a glance. The diagram is regenerated from `CONFIG['edges']` + `CONFIG['agents']` every time the cell runs, so edits to the dict above are reflected immediately.

In [3]:
CONFIG = {   'id': 'chain-9b-temp-lo-seed-random-rep3',
    'description': 'Type-driven linear chain: locator → patcher → reviewer → finalizer. Each agent '
                   'inherits prompts/tools from its type class under evomas/agents/types/ — no '
                   'bespoke Python.',
    'entry': 'locator',
    'end': ['finalizer'],
    'edges': [   {'from': 'locator', 'to': 'patcher'},
                 {'from': 'patcher', 'to': 'reviewer'},
                 {'from': 'reviewer', 'to': 'finalizer'}],
    'agents': {   'locator': {   'class': 'LocatorAgent',
                                 'model': 'ollama/qwen3.5:9b',
                                 'think': False,
                                 'num_ctx': 8192,
                                 'stream': True,
                                 'temperature': 0.0,
                                 'top_k': 40,
                                 'top_p': 0.9,
                                 'min_p': 0,
                                 'repeat_penalty': 1.1,
                                 'repeat_last_n': 64,
                                 'seed': -1,
                                 'num_predict': 512,
                                 'stop': ['</files>'],
                                 'max_iters': 6},
                  'patcher': {   'class': 'PatcherAgent',
                                 'model': 'ollama/qwen3.5:9b',
                                 'think': True,
                                 'num_ctx': 16384,
                                 'stream': True,
                                 'temperature': 0.0,
                                 'top_k': 40,
                                 'top_p': 0.9,
                                 'min_p': 0,
                                 'repeat_penalty': 1.1,
                                 'repeat_last_n': 64,
                                 'seed': -1,
                                 'num_predict': 2048,
                                 'stop': ['</patch>'],
                                 'max_iters': 12,
                                 'fallback': {'enabled': True, 'guarantee_change': True}},
                  'reviewer': {   'class': 'ReviewerAgent',
                                  'model': 'ollama/qwen3.5:9b',
                                  'think': True,
                                  'num_ctx': 4096,
                                  'stream': True,
                                  'temperature': 0.0,
                                  'top_k': 40,
                                  'top_p': 0.9,
                                  'min_p': 0,
                                  'repeat_penalty': 1.1,
                                  'repeat_last_n': 64,
                                  'seed': -1,
                                  'num_predict': 1024,
                                  'stop': ['</review>'],
                                  'max_iters': 6},
                  'finalizer': {   'class': 'HelperProxyAgent',
                                   'model': 'ollama/qwen3.5:9b',
                                   'think': True,
                                   'num_ctx': 4096,
                                   'stream': True,
                                   'temperature': 0.0,
                                   'top_k': 40,
                                   'top_p': 0.9,
                                   'min_p': 0,
                                   'repeat_penalty': 1.1,
                                   'repeat_last_n': 64,
                                   'seed': -1,
                                   'num_predict': 512,
                                   'stop': [],
                                   'max_iters': 4}}}

In [4]:
from IPython.display import Markdown, display

def _topology_mermaid(cfg):
    """Render the topology as a Mermaid flowchart.

    Mirrors what the topology page's cytoscape canvas shows:
    virtual START/END boundary nodes, one node per agent with
    its class as a second-line label, edges directed left-to-
    right. Renders inline in Jupyter Lab + VSCode Jupyter; if
    the cell falls back to plain text the source stays readable.
    """
    lines = ['graph LR']
    lines.append('    START((START))')
    lines.append('    END((END))')
    for name, block in (cfg.get('agents') or {}).items():
        cls = (block or {}).get('class', '') or ''
        label = f'{name}<br/><i>{cls}</i>' if cls else name
        # Backticks break the mermaid parser; strip them.
        label = label.replace('`', '')
        lines.append(f'    {name}["{label}"]')
    entry = cfg.get('entry') or ''
    if entry:
        lines.append(f'    START --> {entry}')
    for e in (cfg.get('edges') or []):
        if isinstance(e, dict) and e.get('from') and e.get('to'):
            lines.append(f'    {e["from"]} --> {e["to"]}')
    end_field = cfg.get('end')
    ends = (
        [end_field] if isinstance(end_field, str) and end_field
        else list(end_field or [])
    )
    # `→ END` only for nodes with no outgoing edges, same
    # rule as `graph_builder.py`.
    out_sources = {e.get('from') for e in (cfg.get('edges') or [])
                   if isinstance(e, dict)}
    for n in ends:
        if n and n not in out_sources:
            lines.append(f'    {n} --> END')
    return '\n'.join(lines)

display(Markdown('```mermaid\n' + _topology_mermaid(CONFIG) + '\n```'))


```mermaid
graph LR
    START((START))
    END((END))
    locator["locator<br/><i>LocatorAgent</i>"]
    patcher["patcher<br/><i>PatcherAgent</i>"]
    reviewer["reviewer<br/><i>ReviewerAgent</i>"]
    finalizer["finalizer<br/><i>HelperProxyAgent</i>"]
    START --> locator
    locator --> patcher
    patcher --> reviewer
    reviewer --> finalizer
    finalizer --> END
```

## 3. Instances

Self-contained: the notebook regenerates its own instances from zero each run. SWE-bench rows get pulled fresh from HuggingFace (cached under `~/.cache/huggingface`); custom rows are reconstructed from the minimal inputs the user added via the Inference page's `+ Custom` modal.

In [5]:
INSTANCE_IDS = [   'custom-EvoMas-evomas-instance-trivial-18757fd',
    'custom-EvoMas-evomas-instance-easy-fcf59bc',
    'custom-EvoMas-evomas-instance-medium-a406a76',
    'custom-EvoMas-evomas-instance-hard-ad94202',
    'custom-EvoMas-evomas-instance-expert-a2e3735']


In [6]:
# `{(subset, split): [ids]}` — one HuggingFace fetch per group.
SWEBENCH_GROUPS = {}


In [7]:
# Custom-instance inputs inlined — no upstream to fetch.
CUSTOM_ROWS = [   {   'instance_id': 'custom-EvoMas-evomas-instance-trivial-18757fd',
        'repo': 'EvoMas/evomas-instance-trivial',
        'base_commit': '18757fdacb59343425bf22a821a10d8978de7f5d',
        'problem_statement': 'evomas-instance-trivial\n'
                             'Synthetic SWE-bench instance for EvoMas APR evaluation - trivial '
                             'difficulty.\n'
                             '\n'
                             'A one-function Python module (`is_even.py`) returns the wrong '
                             'boolean: `n % 2 == 1` should be `n % 2 == 0`. A failing pytest suite '
                             '(`test_is_even.py`) exercises the bug across positive, negative and '
                             'zero inputs.',
        'hints_text': '',
        'subset': 'custom',
        'split': 'custom'},
    {   'instance_id': 'custom-EvoMas-evomas-instance-easy-fcf59bc',
        'repo': 'EvoMas/evomas-instance-easy',
        'base_commit': 'fcf59bcfe0533b786f1b57e63bfdf1163c6905ed',
        'problem_statement': 'evomas-test-instance\n'
                             'Synthetic test repository for EvoMas APR evaluation.\n'
                             '\n'
                             'Contains a simple Python calculator module with a deliberate bug for '
                             'testing automated program repair.',
        'hints_text': '',
        'subset': 'custom',
        'split': 'custom'},
    {   'instance_id': 'custom-EvoMas-evomas-instance-medium-a406a76',
        'repo': 'EvoMas/evomas-instance-medium',
        'base_commit': 'a406a76824b3f74bb4b808a2dc1e7d0aee0f7811',
        'problem_statement': 'evomas-instance-medium\n'
                             'Synthetic SWE-bench instance for EvoMas APR evaluation - medium '
                             'difficulty.\n'
                             '\n'
                             '`rotate.py:rotate_left(arr, n)` slices the input with `arr[n:] + '
                             'arr[:n]`. This works for `n < len(arr)` but silently breaks for `n '
                             '>= len(arr)`: e.g. `rotate_left([1, 2, 3], 3)` returns `[]` instead '
                             'of `[1, 2, 3]`, and `rotate_left([1, 2, 3], 5)` returns `[]` instead '
                             'of `[2, 3, 1]`. The fix is one line - normalize `n` modulo the array '
                             'length before the slice (`n = n % len(arr)`).',
        'hints_text': '',
        'subset': 'custom',
        'split': 'custom'},
    {   'instance_id': 'custom-EvoMas-evomas-instance-hard-ad94202',
        'repo': 'EvoMas/evomas-instance-hard',
        'base_commit': 'ad94202ad8c9f02c2521fda1c7181d1c4af027b9',
        'problem_statement': 'evomas-instance-hard\n'
                             'Synthetic SWE-bench instance for EvoMas APR evaluation - hard '
                             'difficulty.\n'
                             '\n'
                             'Classic Python pitfall: `accumulator.py:accumulate(value, '
                             'history=[])` uses a mutable default argument, so every call without '
                             'an explicit `history` shares the same list object. The test '
                             '`test_independent_default_calls` fails because state leaks across '
                             'calls. The fix is `history=None` + `if history is None: history = '
                             '[]`.',
        'hints_text': '',
        'subset': 'custom',
        'split': 'custom'},
    {   'instance_id': 'custom-EvoMas-evomas-instance-expert-a2e3735',
        'repo': 'EvoMas/evomas-instance-expert',
        'base_commit': 'a2e3735795413732cdd80dc5d0b147e323425748',
        'problem_statement': 'evomas-instance-expert\n'
                             'Synthetic SWE-bench instance for EvoMas APR evaluation - expert '
                             'difficulty.\n'
                             '\n'
                             '`cleanup.py:remove_negatives` mutates the list while iterating over '
                             'it: after `items.pop(i)` every subsequent index shifts down by one '
                             'but `enumerate(items)` keeps marching forward, so consecutive '
                             'negative values get silently skipped. The function appears correct '
                             'line-by-line - only the output values reveal the iterator-semantics '
                             'bug. A correct fix uses a list comprehension, reverse iteration, or '
                             'builds a new list.',
        'hints_text': '',
        'subset': 'custom',
        'split': 'custom'}]


In [8]:
output_dir = RUN_OUTPUT_DIR
output_path = output_dir / 'prediction-chain-9b-temp-lo-seed-random-rep3.jsonl'
INSTANCES_PATH = output_dir / 'instances.jsonl'

selected = []
for (subset, split), ids in SWEBENCH_GROUPS.items():
    print(f'Fetching {len(ids)} {subset}/{split} row(s) from HuggingFace…')
    selected.extend(fetch_swebench_instances(subset, split, instance_ids=ids))
selected.extend(CUSTOM_ROWS)

with INSTANCES_PATH.open('w', encoding='utf-8') as _fh:
    for _row in selected:
        _fh.write(json.dumps(_row, ensure_ascii=False) + '\n')
print(f'Wrote {len(selected)} instance row(s) -> {INSTANCES_PATH}')

_have = {i['instance_id'] for i in selected}
missing = [iid for iid in INSTANCE_IDS if iid not in _have]
if missing:
    print('Missing rows (id not found in HF or in CUSTOM_ROWS):', missing)
print(f'Ready to run {len(selected)} instance(s).')


Wrote 5 instance row(s) -> C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3\instances.jsonl
Ready to run 5 instance(s).


## 4. Inference

Re-runs the EvoMas workflow for each instance with the inlined config. All notebook-produced artefacts (prediction JSONL, evaluation reports, custom-instance sidecar) land under one per-run folder at `notebook-chain-9b-temp-lo-seed-random-rep3/` so they stay grouped together and don't mix with UI/CLI runs in the repo's `results/` tree.

In [9]:
from evomas.exceptions.errors import OllamaMemoryError

predictions = []
with open(output_path, 'w', encoding='utf-8') as out:
    for inst in selected:
        iid = inst['instance_id']
        print(f'--- {iid} ---')
        try:
            patch = run_evomas(inst, config=CONFIG)
        except OllamaMemoryError as exc:
            print(f'Ollama OOM; aborting: {exc}')
            break
        except Exception as exc:
            print(f'run failed on {iid}: {exc}')
            patch = ''
        rec = {
            'instance_id': iid,
            'model_patch': patch,
            'model_name_or_path': 'evomas-notebook',
        }
        predictions.append(rec)
        out.write(json.dumps(rec) + '\n')
print(f'Wrote {len(predictions)} prediction(s) to {output_path}.')


2026-06-07 13:54:41,651 [WARNING] weave.trace.op: Warning: Traces will not be logged. Call weave.init to log your traces to a project.
 (subsequent messages of this type will be suppressed)


2026-06-07 13:54:41,652 [INFO] evomas.core.workflow.runner: === running custom-EvoMas-evomas-instance-trivial-18757fd with inline config (id=chain-9b-temp-lo-seed-random-rep3) ===


--- custom-EvoMas-evomas-instance-trivial-18757fd ---


2026-06-07 13:54:41,801 [INFO] evomas.utils.workspace: reusing workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\custom-EvoMas-evomas-instance-trivial-18757fd (HEAD=18757fdacb59343425bf22a821a10d8978de7f5d)


2026-06-07 13:54:42,007 [INFO] evomas.core.workflow.runner: graph runtime: 4 agents x 2 max revisits => recursion_limit=8


...

2026-06-07 14:25:41,337 [INFO] evomas.models.langchain_ollama_model: [finalizer|resp ] patch accepted: The iterator-semantics bug in cleanup.py:remove_negatives was fixed by replacing the mutating loop with a list comprehension, lint check passed.


2026-06-07 14:25:41,338 [INFO] evomas.models.langchain_ollama_model: [finalizer] tokens in=983 out=104 total=1087


2026-06-07 14:25:41,338 [INFO] evomas.agents.finalizer: [finalizer] no tool calls — stopping loop


2026-06-07 14:25:41,386 [INFO] evomas.core.workflow.runner: === custom-EvoMas-evomas-instance-expert-a2e3735 done: 517-char patch | tokens in=47550 out=2785 total=50335 ===


Wrote 5 prediction(s) to C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3\prediction-chain-9b-temp-lo-seed-random-rep3.jsonl.


## 5. Evaluation

Runs `scripts/evaluation/apply_and_test.py` — the evaluator chosen at notebook-generation time (the `--evaluator` flag on `evomas notebook`). Forwards through the unified evaluator CLI contract: `--predictions / --instances / --report-dir / --run-id / --model`. Output lands at `<report-dir>/{model}.{run-id}.json` plus per-instance folders under `<report-dir>/logs/run_evaluation/<run-id>/<model>/<instance>/`.

In [10]:
EVALUATOR_STEM = 'apply_and_test'
EVALUATOR_NEEDS_WSL = False

first = selected[0] if selected else None
SUBSET = (first or {}).get('subset', 'lite')
SPLIT  = (first or {}).get('split',  'dev')

eval_report_dir = output_dir

import platform
from evomas.paths import BASE_DIR as _BASE_DIR
_script = _BASE_DIR / 'scripts' / 'evaluation' / f'{EVALUATOR_STEM}.py'
if EVALUATOR_NEEDS_WSL and platform.system() == 'Windows':
    from evomas.utils.paths import to_wsl
    cmd = [
        'wsl', '--', 'python3', to_wsl(str(_script)),
        '--predictions', to_wsl(str(output_path)),
        '--instances',   to_wsl(str(INSTANCES_PATH)),
        '--report-dir',  to_wsl(str(eval_report_dir)),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
else:
    cmd = [
        sys.executable, str(_script),
        '--predictions', str(output_path),
        '--instances',   str(INSTANCES_PATH),
        '--report-dir',  str(eval_report_dir),
        '--run-id',      f'notebook-{SUBSET}-{SPLIT}',
        '--model',       'evomas-notebook',
    ]
print(f'Evaluating via {EVALUATOR_STEM}.py')
print('+ ' + ' '.join(cmd))

eval_proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, encoding='utf-8', errors='replace',
)
assert eval_proc.stdout is not None
for line in eval_proc.stdout:
    print(line, end='')
eval_proc.wait()
print(f'\n[evaluation finished with exit code {eval_proc.returncode}]')

logs_root = eval_report_dir / 'logs' / 'run_evaluation'
if logs_root.is_dir():
    print('\nPer-instance artifacts:')
    for inst_dir in sorted(logs_root.rglob('*/')):
        if (inst_dir / 'report.json').is_file():
            print(f'  {inst_dir}')
for summary in sorted(eval_report_dir.glob('*.json')):
    print(f'Summary: {summary}')


Evaluating via apply_and_test.py
+ C:\Users\XF\.evomas-venv\Scripts\python.exe C:\Users\XF\Desktop\TFG\EvoMas\scripts\evaluation\apply_and_test.py --predictions C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3\prediction-chain-9b-temp-lo-seed-random-rep3.jsonl --instances C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3\instances.jsonl --report-dir C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3 --run-id notebook-custom-custom --model evomas-notebook
2026-06-07 14:25:41,569 - INFO - Evaluating 5 instance(s)
2026-06-07 14:25:41,569 - INFO - -- custom-EvoMas-evomas-instance-trivial-18757fd --
2026-06-07 14:25:41,570 - INFO - Cleaning workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\EvoMas__evomas-instance-trivial (base_commit=18757fda)


+---------- custom-EvoMas-evomas-instance-trivial-18757fd  RESOLVED ----------+
| patch: applied                                                              |
| rule:  pytest_returncode==0                                                 |
| pytest returncode: 0                                                        |
|                                                                             |
+-----------------------------------------------------------------------------+
2026-06-07 14:25:43,056 - INFO - -- custom-EvoMas-evomas-instance-easy-fcf59bc --
2026-06-07 14:25:43,058 - INFO - Cleaning workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\EvoMas__evomas-instance-easy (base_commit=fcf59bcf)


+--------- custom-EvoMas-evomas-instance-easy-fcf59bc  NOT RESOLVED ----------+
| patch: applied                                                              |
| rule:  pytest_returncode==0                                                 |
| pytest returncode: 2                                                        |
|                                                                             |
+-----------------------------------------------------------------------------+
2026-06-07 14:25:44,817 - INFO - -- custom-EvoMas-evomas-instance-medium-a406a76 --
2026-06-07 14:25:44,817 - INFO - Cleaning workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\EvoMas__evomas-instance-medium (base_commit=a406a768)


+-------- custom-EvoMas-evomas-instance-medium-a406a76  NOT RESOLVED ---------+
| patch: applied                                                              |
| rule:  pytest_returncode==0                                                 |
| pytest returncode: 1                                                        |
|                                                                             |
+-----------------------------------------------------------------------------+
2026-06-07 14:25:46,434 - INFO - -- custom-EvoMas-evomas-instance-hard-ad94202 --
2026-06-07 14:25:46,434 - INFO - Cleaning workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\EvoMas__evomas-instance-hard (base_commit=ad94202a)


+----------- custom-EvoMas-evomas-instance-hard-ad94202  RESOLVED ------------+
| patch: applied                                                              |
| rule:  pytest_returncode==0                                                 |
| pytest returncode: 0                                                        |
|                                                                             |
+-----------------------------------------------------------------------------+
2026-06-07 14:25:47,909 - INFO - -- custom-EvoMas-evomas-instance-expert-a2e3735 --
2026-06-07 14:25:47,910 - INFO - Cleaning workspace at C:\Users\XF\AppData\Local\Temp\evomas_workspace\EvoMas__evomas-instance-expert (base_commit=a2e37357)


+---------- custom-EvoMas-evomas-instance-expert-a2e3735  RESOLVED -----------+
| patch: applied                                                              |
| rule:  pytest_returncode==0                                                 |
| pytest returncode: 0                                                        |
|                                                                             |
+-----------------------------------------------------------------------------+
2026-06-07 14:25:49,395 - INFO - Run summary -> C:\Users\XF\Desktop\TFG\EvoMas\notebooks\notebook-chain-9b-temp-lo-seed-random-rep3\evomas-notebook.notebook-custom-custom.json
+-----------------------------------------------------------------------------+
| Resolved 3/5 instances                                                      |
+-----------------------------------------------------------------------------+

[evaluation finished with exit code 0]

Per-instance artifacts:
  C:\Users\XF\Desktop\TFG\EvoMas\notebo